In [1]:
!git clone https://github.com/kdjoumessi/Self-Explainable-CNN-Transformer /kaggle/working/code
%cd /kaggle/working/code
!pip install -q git+https://github.com/wielandbrendel/bag-of-local-features-models.git
!pip install -q omegaconf munch  # if not already available

Cloning into '/kaggle/working/code'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 124 (delta 26), reused 117 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.77 MiB | 18.13 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/kaggle/working/code
  Preparing metadata (setup.py) ... done


In [3]:
!rm -rf /kaggle/working/train_raw

In [4]:
import os, cv2, subprocess
import numpy as np
from pathlib import Path
from tqdm import tqdm

BASE       = "/kaggle/input/competitions/diabetic-retinopathy-detection"
TEMP_DIR   = "/kaggle/working/temp_jpegs"
OUTPUT_DIR = "/kaggle/working/kaggle_512"
BATCH_SIZE = 500   # lower = less temp disk use, more 7z calls; 500 is a good balance

os.makedirs(TEMP_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

def crop_and_resize(img, size=512):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return cv2.resize(img, (size, size))
    x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
    return cv2.resize(img[y:y+h, x:x+w], (size, size))

# 1. Get the full list of files inside the archive (no extraction yet)
result = subprocess.run(
    ["7z", "l", "-ba", f"{BASE}/train.zip.001"],
    capture_output=True, text=True
)
all_files = [
    line.strip().split()[-1]
    for line in result.stdout.splitlines()
    if line.strip().endswith(".jpeg")
]
print(f"Archive contains {len(all_files)} JPEG files")

# 2. Skip files already processed (safe to re-run this cell)
already_done = {p.stem for p in Path(OUTPUT_DIR).glob("*.png")}
to_process   = [f for f in all_files if Path(f).stem not in already_done]
print(f"{len(already_done)} already processed, {len(to_process)} remaining")

# 3. Extract → process → delete in batches
for i in tqdm(range(0, len(to_process), BATCH_SIZE), desc="Batches"):
    batch = to_process[i : i + BATCH_SIZE]

    # Extract this batch only
    subprocess.run(
        ["7z", "e", f"{BASE}/train.zip.001", f"-o{TEMP_DIR}", "-y"] + batch,
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

    # Process and immediately delete each raw JPEG
    for fpath in batch:
        fname = os.path.basename(fpath)
        src   = os.path.join(TEMP_DIR, fname)
        if not os.path.exists(src):
            continue
        img = cv2.imread(src)
        if img is not None:
            processed = crop_and_resize(img)
            cv2.imwrite(os.path.join(OUTPUT_DIR, Path(fname).stem + ".png"), processed)
        os.remove(src)  # delete raw JPEG immediately — keeps disk free

print(f"\nDone. {len(list(Path(OUTPUT_DIR).glob('*.png')))} PNGs in {OUTPUT_DIR}")

Archive contains 35126 JPEG files
0 already processed, 35126 remaining


Batches: 100%|██████████| 71/71 [51:33<00:00, 43.57s/it]


Done. 35126 PNGs in /kaggle/working/kaggle_512


In [13]:
func_path = "/kaggle/working/code/utils/func.py"
with open(func_path) as f:
    content = f.read()

old = """def load_save_paths(cfg):
    timestamp_str = datetime.now().strftime("%d-%m-%Y_%H:%M:%S")  
    save_path_model = 'Outputs/tmp/BagNet_SA' if cfg.base.test else f'Outputs/BagNet_SA/{cfg.base.dataset}'
    save_path = os.path.join(os.path.expanduser('~'), save_path_model, timestamp_str) 
    return save_path"""

new = """def load_save_paths(cfg):
    timestamp_str = datetime.now().strftime("%d-%m-%Y_%H:%M:%S")
    if cfg.dset.save_path:
        save_path = os.path.join(cfg.dset.save_path, timestamp_str)
    else:
        save_path_model = 'Outputs/tmp/BagNet_SA' if cfg.base.test else f'Outputs/BagNet_SA/{cfg.base.dataset}'
        save_path = os.path.join(os.path.expanduser('~'), save_path_model, timestamp_str)
    return save_path"""

content = content.replace(old, new)
with open(func_path, "w") as f:
    f.write(content)
print("Patched.")

Patched.


In [14]:
import yaml

paths_file = "/kaggle/working/code/configs/paths.yaml"
with open(paths_file) as f:
    cfg = yaml.safe_load(f)

cfg["Fundus"]["root"] = "/kaggle/working"
cfg["Fundus"]["data_dir"] = "kaggle_512"

with open(paths_file, "w") as f:
    yaml.dump(cfg, f)

print("paths.yaml updated.")

paths.yaml updated.


In [15]:
default_file = "/kaggle/working/code/configs/default.yaml"
with open(default_file) as f:
    cfg = yaml.safe_load(f)

cfg["train"]["epochs"] = 50        # full=70, quick test=10
cfg["train"]["batch_size"] = 8     # P100 can handle 8 at 512x512
cfg["train"]["num_workers"] = 4
cfg["dset"]["save_path"] = "/kaggle/working/outputs"

with open(default_file, "w") as f:
    yaml.dump(cfg, f)

print("default.yaml updated.")

default.yaml updated.


In [16]:
!pip install -q munch
!pip install git+https://github.com/wielandbrendel/bag-of-local-features-models.git


  Cloning https://github.com/wielandbrendel/bag-of-local-features-models.git to /tmp/pip-req-build-nlzgv6ac
  Running command git clone --filter=blob:none --quiet https://github.com/wielandbrendel/bag-of-local-features-models.git /tmp/pip-req-build-nlzgv6ac
  Resolved https://github.com/wielandbrendel/bag-of-local-features-models.git to commit 8493fb442c9b9ccfeaa970c950ca67832017f072
  Preparing metadata (setup.py) ... done


In [17]:
# Temporarily train on 50 samples to confirm everything works
import yaml
with open(default_file) as f: cfg = yaml.safe_load(f)
cfg["base"]["sample"] = 50
with open(default_file, "w") as f: yaml.dump(cfg, f)

!cd /kaggle/working/code && python main.py

# Reset sample to 0 for full training
with open(default_file) as f: cfg = yaml.safe_load(f)
cfg["base"]["sample"] = 0
with open(default_file, "w") as f: yaml.dump(cfg, f)
print("Smoke test passed — ready for full training.")

2026-05-09 10:52:38.395503: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778323958.418868     594 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778323958.427096     594 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778323958.451201     594 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778323958.451264     594 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778323958.451273     594 computation_placer.cc:177] computation placer alr

In [18]:
# Cell 5 — set full training config:


import yaml
default_file = "/kaggle/working/code/configs/default.yaml"
with open(default_file) as f:
    cfg = yaml.safe_load(f)

cfg["train"]["epochs"]      = 50
cfg["train"]["batch_size"]  = 8
cfg["train"]["num_workers"] = 4
cfg["base"]["sample"]       = 0
cfg["base"]["test"]         = False   # important — smoke test sets this to True
cfg["dset"]["save_path"]    = "/kaggle/working/outputs/bagnet_drsa"

with open(default_file, "w") as f:
    yaml.dump(cfg, f)
print("Ready.")

Ready.


In [20]:
import pandas as pd
from pathlib import Path

PNG_DIR  = "/kaggle/working/kaggle_512"
CODE_DIR = "/kaggle/working/code"

existing = {p.stem for p in Path(PNG_DIR).glob("*.png")}
print(f"PNGs on disk: {len(existing)}")

for split in ["train", "val", "test"]:
    csv_path = f"{CODE_DIR}/files/csv/Kaggle/kaggle_gradable_{split}.csv"
    df = pd.read_csv(csv_path)
    before = len(df)
    df = df[df["filename"].apply(lambda x: Path(x).stem in existing)]
    after = len(df)
    df.to_csv(csv_path, index=False)
    print(f"{split}: {before} → {after} rows ({before - after} missing dropped)")

PNGs on disk: 35126
train: 34350 → 13531 rows (20819 missing dropped)
val: 4617 → 1857 rows (2760 missing dropped)
test: 6956 → 2750 rows (4206 missing dropped)


In [ ]:
!cd /kaggle/working/code && python main.py


2026-05-09 11:30:58.073719: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778326258.098245     887 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778326258.105961     887 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778326258.127245     887 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778326258.127282     887 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778326258.127286     887 computation_placer.cc:177] computation placer alr

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/outputs

In [3]:
import pandas as pd
df = pd.read_csv("/kaggle/working/outputs/bagnet_drsa/09-05-2026_11:31:02/summarize.csv")
print(df[["train_loss", "train_acc", "val_loss", "val_acc", "val_kappa"]].to_string())

    train_loss  train_acc  val_loss   val_acc  val_kappa
0     0.977412   0.729746  0.852188  0.737749   0.000000
1     0.867219   0.730411  0.813564  0.737749   0.000000
2     0.833011   0.730411  0.773224  0.737749   0.000000
3     0.767935   0.737951  0.696660  0.780291   0.514484
4     0.704262   0.775651  0.598698  0.814216   0.638196
5     0.652724   0.795240  0.599564  0.812601   0.649286
6     0.628387   0.804923  0.570478  0.827141   0.732610
7     0.607780   0.809506  0.562455  0.829295   0.773535
8     0.591640   0.816603  0.534040  0.838449   0.761929
9     0.584438   0.821038  0.551690  0.829295   0.705992
10    0.573062   0.823847  0.532552  0.835218   0.739387
11    0.565877   0.825399  0.528655  0.831449   0.734342
12    0.559734   0.826138  0.529565  0.833064   0.732136
13    0.554617   0.828134  0.522783  0.830372   0.730950
14    0.547033   0.831387  0.505435  0.842757   0.778923
15    0.537605   0.829095  0.521012  0.835218   0.775228
16    0.526670   0.834861  0.55

In [4]:
import shutil
shutil.make_archive(
    '/kaggle/working/bagnet_drsa_best_model',  # output zip name
    'zip',
    '/kaggle/working/outputs/bagnet_drsa/09-05-2026_11:31:02'  # folder to zip
)
print("Done. File: /kaggle/working/bagnet_drsa_best_model.zip")

Done. File: /kaggle/working/bagnet_drsa_best_model.zip


In [5]:
import shutil
shutil.make_archive(
    '/kaggle/working/bagnet_kappa_only',
    'zip',
    '/kaggle/working/outputs/bagnet_drsa/09-05-2026_11:31:02',
    'best_validation_weights_kappa.pt'   # only this file
)
print("Done — ~550MB zip")

Done — ~550MB zip


In [6]:
# Mount Google Drive
from google.colab import drive   # won't work in Kaggle — use this instead:

!pip install -q pydrive2

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

file = drive.CreateFile({'title': 'best_validation_weights_kappa.pt'})
file.SetContentFile('/kaggle/working/outputs/bagnet_drsa/09-05-2026_11:31:02/best_validation_weights_kappa.pt')
file.Upload()
print(f"Uploaded! File ID: {file['id']}")


Uploaded! File ID: 1M9z-NKQuqUhLanNSmVZzbKBnCJszyNij
